# Scenario 1

A small report over an events directory. It takes longer than the data volume suggests.

Find out why, using the Spark UI (http://localhost:4040). Write your diagnosis in chat as:

`Symptom → Where I looked → Evidence (numbers) → Hypothesis → What I would change → How I would verify`

In [1]:
import time
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("m6-scenario-1").getOrCreate()

schema = "event_id BIGINT, user_id INT, event_type STRING, amount DOUBLE, payload STRING"
events = spark.read.schema(schema).csv("/opt/data/m6/events/*")

report = (
    events.filter(F.col("amount") > 10)
    .groupBy("event_type")
    .agg(F.count("*").alias("events"), F.sum("amount").alias("total_amount"))
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/01 09:54:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/01 09:55:03 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /opt/data/m6/events/*.
java.io.FileNotFoundException: File /opt/data/m6/events/* does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStr

In [3]:
for f in sorted(events.inputFiles()):
    print(f)

file:///opt/data/m6/events/part_a/part-00000-6307a289-5a9e-4626-8493-253687463eb6-c000.csv
file:///opt/data/m6/events/part_a/part-00001-6307a289-5a9e-4626-8493-253687463eb6-c000.csv
file:///opt/data/m6/events/part_a/part-00002-6307a289-5a9e-4626-8493-253687463eb6-c000.csv
file:///opt/data/m6/events/part_a/part-00003-6307a289-5a9e-4626-8493-253687463eb6-c000.csv
file:///opt/data/m6/events/part_a/part-00004-6307a289-5a9e-4626-8493-253687463eb6-c000.csv
file:///opt/data/m6/events/part_a/part-00005-6307a289-5a9e-4626-8493-253687463eb6-c000.csv
file:///opt/data/m6/events/part_a/part-00006-6307a289-5a9e-4626-8493-253687463eb6-c000.csv
file:///opt/data/m6/events/part_a/part-00007-6307a289-5a9e-4626-8493-253687463eb6-c000.csv
file:///opt/data/m6/events/part_a/part-00008-6307a289-5a9e-4626-8493-253687463eb6-c000.csv
file:///opt/data/m6/events/part_a/part-00009-6307a289-5a9e-4626-8493-253687463eb6-c000.csv
file:///opt/data/m6/events/part_a/part-00010-6307a289-5a9e-4626-8493-253687463eb6-c000.csv

In [2]:
t0 = time.perf_counter()
report.write.format("noop").mode("overwrite").save()
print(f"report: {time.perf_counter() - t0:.1f}s")

[Stage 0:====================================================>      (8 + 1) / 9]

report: 11.0s


## Stop before the next scenario

In [ ]:
spark.stop()